In [1]:
import polars as pl
from sqlalchemy import create_engine

In [2]:
uri = "postgresql://p8:p8@localhost:5433/meteoforecast"

In [15]:
# --- xlsx ---

df_ichtegem = pl.read_database_uri("SELECT * FROM raw.weather_ichtegem", uri)
df_la_madeleine = pl.read_database_uri("SELECT * FROM raw.weather_la_madeleine", uri)

# --- json (non deplier) ---

df_infoclimat = pl.read_database_uri("SELECT * FROM raw.infoclimat", uri)
df_meteo = pl.read_database_uri("SELECT * FROM raw.open_meteo_roubaix", uri)

In [16]:
df_ichtegem.describe()

statistic,_airbyte_raw_id,_airbyte_extracted_at,_airbyte_meta,_airbyte_generation_id,UV,Gust,Time,Wind,Solar,Speed,Humidity,Pressure,Dew_Point,Temperature,Precip__Rate_,Precip__Accum_
str,str,str,str,f64,f64,str,str,str,str,str,str,str,str,str,str,str
"""count""","""1899""","""1899""","""1899""",1899.0,1899.0,"""1899""","""1899""","""1892""","""1899""","""1899""","""1899""","""1899""","""1899""","""1899""","""1899""","""1899"""
"""null_count""","""0""","""0""","""0""",0.0,0.0,"""0""","""0""","""7""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0"""
"""mean""",null,"""2026-07-18 15:01:20.945760+00:…",null,3.0,0.627699,null,null,null,null,null,null,null,null,null,null,null
"""std""",null,null,null,0.0,1.125381,null,null,null,null,null,null,null,null,null,null,null
"""min""","""019f75bf-0ce6-7382-adb1-032640…","""2026-07-18 15:01:20+00:00""","""{""changes"":[],""sync_id"":8}""",3.0,0.0,"""0.0 mph""","""00:04:00""","""ENE""","""0.1 w/m²""","""0.0 mph""","""55 %""","""29.27 in""","""36.0 °F""","""36.5 °F""","""0.00 in""","""0.00 in"""
"""25%""",null,"""2026-07-18 15:01:20+00:00""",null,3.0,0.0,null,null,null,null,null,null,null,null,null,null,null
"""50%""",null,"""2026-07-18 15:01:21+00:00""",null,3.0,0.0,null,null,null,null,null,null,null,null,null,null,null
"""75%""",null,"""2026-07-18 15:01:22+00:00""",null,3.0,1.0,null,null,null,null,null,null,null,null,null,null,null
"""max""","""019f75bf-157f-7e60-9be4-44253a…","""2026-07-18 15:01:22+00:00""","""{""changes"":[],""sync_id"":8}""",3.0,5.0,"""9.9 mph""","""23:59:00""","""West""","""98 w/m²""","""9.9 mph""","""98 %""","""29.98 in""","""58.0 °F""","""66.4 °F""","""0.35 in""","""0.18 in"""


In [5]:
def renseigne(df): # Fonction de profiling adapter à polars
    n = df.height
    nulls = df.null_count().row(0)
    profil = pl.DataFrame({
        'colonne': df.columns, 
        'type':[str(t)for t in df.dtypes],
        'longueur_max':[df[c].cast(pl.Utf8).str.len_chars().max() for c in df.columns],
        'non_null':[n - x for x in nulls],
        'nan':list(nulls),
        "% nan": [round(x / n * 100, 2) for x in nulls],
        'valeur_unique':[df[c].n_unique()for c in df.columns]
    })
    return profil

#cols_airbnb = renseigne(df) # Instance le df profilé
# cols_airbnb.write_csv('../docs/cols_airbnb.csv') # Produit un csv du profil
#display(cols_airbnb) # Affiche le résultat de la fonctioin de profiling

In [ ]:
df_ichtegem.columns


['_airbyte_raw_id',
 '_airbyte_extracted_at',
 '_airbyte_meta',
 '_airbyte_generation_id',
 'UV',
 'Gust',
 'Time',
 'Wind',
 'Solar',
 'Speed',
 'Humidity',
 'Pressure',
 'Dew_Point',
 'Temperature',
 'Precip__Rate_',
 'Precip__Accum_']

In [19]:
df_la_madeleine.columns

['_airbyte_raw_id',
 '_airbyte_extracted_at',
 '_airbyte_meta',
 '_airbyte_generation_id',
 'UV',
 'Gust',
 'Time',
 'Wind',
 'Solar',
 'Speed',
 'Humidity',
 'Pressure',
 'Dew_Point',
 'Temperature',
 'Precip__Rate_',
 'Precip__Accum_']

In [14]:
renseigne(df_ichtegem)

colonne,type,longueur_max,non_null,nan,% nan,valeur_unique
str,str,i64,i64,i64,f64,i64
"""_airbyte_raw_id""","""String""",36,1908,0,0.0,1908
"""_airbyte_extracted_at""","""Datetime(time_unit='us', time_…",32,1908,0,0.0,3
"""_airbyte_meta""","""String""",26,1908,0,0.0,1
"""_airbyte_generation_id""","""Int64""",1,1908,0,0.0,1
"""UV""","""Decimal(precision=38, scale=10…",12,1908,0,0.0,5
…,…,…,…,…,…,…
"""Pressure""","""String""",8,1908,0,0.0,70
"""Dew_Point""","""String""",7,1908,0,0.0,180
"""Temperature""","""String""",7,1908,0,0.0,242


In [8]:
df_infoclimat.schema

Schema([('_airbyte_raw_id', String),
        ('_airbyte_extracted_at', Datetime(time_unit='us', time_zone='UTC')),
        ('_airbyte_meta', String),
        ('_airbyte_generation_id', Int64),
        ('data', String),
        ('errors', String),
        ('hourly', String),
        ('status', String),
        ('metadata', String),
        ('stations', String)])

In [ ]:
import json
mesures = json.loads(df_infoclimat["hourly"][0])["07015"]   # tableau de mesures
df_i = pl.DataFrame(mesures) 
renseigne(df_i)


colonne,type,longueur_max,non_null,nan,% nan,valeur_unique
str,str,i64,i64,i64,f64,i64
"""dh_utc""","""String""",19,60,0,0.0,60
"""humidite""","""String""",3,60,0,0.0,28
"""id_station""","""String""",5,60,0,0.0,1
"""nebulosite""","""String""",1,60,0,0.0,4
"""neige_au_sol""","""Null""",null,0,60,100.0,1
…,…,…,…,…,…,…
"""temps_omm""","""String""",2,9,51,85.0,5
"""vent_direction""","""String""",3,60,0,0.0,18
"""vent_moyen""","""String""",4,60,0,0.0,7


In [20]:
df_ichtegem["Time"].head(10)

Time
str
"""00:04:00"""
"""00:09:00"""
"""00:14:00"""
"""00:19:00"""
"""00:24:00"""
"""00:29:00"""
"""00:34:00"""
"""00:39:00"""
"""00:44:00"""


In [21]:
df_ichtegem["Time"].n_unique()


291

In [23]:
df_la_madeleine["Time"].to_list()[268:278]


['22:24:00',
 '22:29:00',
 '22:34:00',
 '22:39:00',
 '22:44:00',
 '22:49:00',
 '22:54:00',
 '22:59:00',
 '23:04:00',
 '23:09:00']